In [229]:
model = "llama3.2:1B"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [230]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = "llama3.2:1B"

In [231]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="llama3.2:1B")


In [232]:

from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection(name="my_collection")


# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(collection_name="my_collection", embedding_function=embedding_model, client=client)

In [233]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.documents import Document

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)

# Lade Dokumente und speichere sie in Chroma
try:
    docs = []
    urls = [
        "https://de.wikipedia.org/wiki/Maschinelles_Lernen",
    ]
    for url in urls:
        docs.extend(WebBaseLoader(url).load())
except:
    # Fallback: verwende Mock-Dokumente wenn WebBaseLoader nicht funktioniert
    docs = [
        Document(page_content="""Maschinelles Lernen ist ein Teilgebiet der künstlichen Intelligenz. 
        Es ermöglicht Computern, aus Daten zu lernen und ihre Leistung zu verbessern.
        Quellen: https://de.wikipedia.org/wiki/Maschinelles_Lernen""", metadata={"source": "Wikipedia ML"})
    ]

# Speichere Dokumente in Chroma Vector DB
vector_db_from_client.add_documents(docs)

# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = prompt | ChatOllama(model=model) | StrOutputParser()

result = chain.invoke({"docs": format_docs(docs)})
print(result)

Der Artikel "Maschinelles Lernen" beschreibt die Grundlagen und Anwendungen von Maschinellem Lernen, einer Technologie zur Analyse und Vorhersage von Daten. Es handelt sich um eine Zusammenfassung der wichtigsten Aspekte des Themas.

**Definition**

Maschinelles Lernen ist ein Bereich der Informatik, der sich mit dem Einsatz von Mustererkennung, Verarbeitung von Daten und Fähigkeit zur Klassifizierung von Daten auf Basis von Algorithmen und Methoden beschäftigt. Es umfasst auch die Entwicklung von Algorithmen und Modelle für das Trainingsprozess, um künstliche Intelligenz (KI) zu implementieren.

**Bereiche**

Maschinelles Lernen kann in verschiedene Bereiche unterteilt werden:

1.  **Klassifizierung**: Hier werden Muster in Daten identifiziert und verwendet, um Vorhersagen für neue unvorhergesehene Daten vorzunehmen.
2.  **Regressionsmodellierung**: Hier werden die Beziehungen zwischen Variablen ausgewertet und verwendet, um Vorhersagen zu treffen.
3.  **Neuralnetze**: Hier werden kom

In [234]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query, k=5)

print(docs)

[Document(id='ml_doc_1', metadata={}, page_content='Machine learning is a branch of artificial intelligence that focuses on \n    building applications that learn from data and improve their performance over time without \n    being explicitly programmed. It involves algorithms and statistical models that enable \n    computers to learn from and make decisions or predictions based on data.\n\n    Key types include supervised learning (with labeled data), unsupervised learning (pattern discovery),\n    and reinforcement learning (learning through interaction). Common applications include image recognition,\n    natural language processing, recommendation systems, and predictive analytics.'), Document(id='d18f04d0-fd7d-4132-ae8f-c7cb4f3fba22', metadata={'source': 'https://de.wikipedia.org/wiki/Maschinelles_Lernen', 'language': 'de', 'title': 'Maschinelles Lernen – Wikipedia'}, page_content="\n\n\n\nMaschinelles Lernen – Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\

In [235]:
chain.invoke({"docs": format_docs(docs)})

'Maschinelles Lernen ist eine disziplin der Informatik, die sich mit der Entwicklung von Systemen, die auf mangelndem oder unbewussten Wissen basieren, arbeiten. Es umfasst das Training von Maschinen, um bestimmte Aufgaben zu erfüllen, und wird oft als Ersatz für menschliche Arbeit in Bereichen wie Wirtschaft, Gesundheit und Bildung verwendet.\n\nDas Konzept des maschinellen Lernens wurde 1943 von John McCarthy entwickelt, einer der Pioniere des informatischen Rechners. Zu Beginn des 20. Jahrhunderts entwickelten Mathematik-Studenten unter der Anleitung von William Shockley, einem Entdecker der transistorisierten elektronischen Schaltkreise, die erste Maschine, die auf dem Lernen basierte.\n\nBereits 1951 präsentierten die ersten maschinellen Witten und künstlichen Intelligenzen (KI) eine grundlegende Entwicklung des maschinellen Lernens. Die erste KI-Software war das Massachusetts Institute of Technology (MIT)-Massachusetts Institute of Technology (MIT) 0/0, ein Computerprogramm, das 

In [236]:
# Simple stream the chain output
for chunk in chain.stream({"docs": format_docs(docs)}):
    print(chunk, end="", flush=True)

Maschinelles Lernen ist ein Bereich der Wissenschaft, der sich mit der Entwicklung von Algorithmen und Kombination aus Datenanalysen, Statistik, Mathematik und Ingenieurwesen widmet. Es umfasst die Analyse von Daten in Form von Beispielen (Exempeln), damit das Verständnis vertrauenswürdiger Informationen verbessert werden kann.

Die Entwicklung von Algorithmen ermöglicht den automatischen Aufbau von Systemen, die ohne menschliche Beteiligung Daten analysieren und Entscheidungen treffen können. Solche Algorithmen können in verschiedenen Bereichen eingesetzt werden:

1. **Klassifizierung**: Die Zuordnung einer Variable (wie Text, Bild oder Zahl) zu einem bestimmten Kategorien-Set.
2. **Regressionsanalyse**: Die Abhängigkeit zwischen zwei oder mehr Variablen.
3. **Funktionale Präsentation von Daten (SPDE)**: Eine Methode zur Visualisierung von Daten, um sie leichter verständlich zu machen.

Einige der wichtigsten Aspekte des Maschinelles Lernens sind:

- **Klassefähigkeit**: Die Fähigkeit

In [237]:
# More complex async event streaming
async for event in chain.astream_events({"docs": format_docs(docs)}, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Die Seite behandelt das Thema Maschinelles Lernen. Maschinelles Lernen, auch bekannt als Künstliche Intelligenz (KI), ist eine Form der Computer-Ähnlichkeit, bei der Computer Programme mit menschlichen Fähigkeiten trainiert werden können, um komplexe Aufgaben zu erfüllen.

Einige wichtige Aspekte von Maschinelles Lernen sind:

* **Training**: Das Training eines Modells bedeutet, dass es auf einer Menge von Daten trainiert wird, die es verbessern kann, indem es bestimmte Fähigkeiten lernt.
* **Verstärkung der Algorithmen**: Durch das Training können die Algorithmen stärker werden und so besser auf neue Daten ausgerichtet werden können.
* **Künstliche Intelligenz**: KI-Systeme können komplexe Aufgaben wie Schreiben, Bilderkennung, Sprachverständnis und Entscheidungsfindung mit hoher Genauigkeit erfüllen.

Einige Beispiele für Anwendungen von Maschinelles Lernen sind:

* **Bilderkennung**: KI-Systeme können Bilder identifizieren und analysieren.
* **Schreiben**: KI-Systeme können Texte sc

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [238]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever(search_kwargs={"k": 5})

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | ChatOllama(model=model)
    | StrOutputParser()
)

In [239]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000020DBF3356D0>, search_kwargs={'k': 5})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(model='llama3.2:1B')
| StrOutputParser()

In [ ]:
# Prüfe explizit, welche Dokumente eingebunden sind
print("=== Überprüfung der eingebundenen Daten ===\n")

# 1. Anzahl der Dokumente in der Collection
count = collection.count()
print(f"Anzahl Dokumente in Chroma: {count}\n")

# 2. Direkte Abfrage des Retrievers
retrieved_docs = retriever.invoke("Maschinelles Lernen")
print(f"Anzahl abgerufener Dokumente: {len(retrieved_docs)}\n")

# 3. Quellen und Inhalte der Dokumente anzeigen
print("=== Inhalte der abgerufenen Dokumente ===\n")
for i, doc in enumerate(retrieved_docs):
    print(f"Dokument {i+1}:")
    print(f"Quelle: {doc.metadata}")
    print(f"Inhalt (erste 200 Zeichen): {doc.page_content[:200]}...")
    print("-" * 80 + "\n")

# Jetzt die Frage für die nächsten Zellen
question = "Was ist Maschinelles Lernen und welche Technologien werden dafür verwendet?"

# Invoke the chain
result = qa_rag_chain.invoke(question)
print("\n=== RAG Chain Antwort ===\n")
print(result)

=== Überprüfung der eingebundenen Daten ===

Anzahl Dokumente in Chroma: 3

Anzahl abgerufener Dokumente: 3

=== Inhalte der abgerufenen Dokumente ===

Dokument 1:
Quelle: {}
Inhalt (erste 200 Zeichen): Machine learning is a branch of artificial intelligence that focuses on 
    building applications that learn from data and improve their performance over time without 
    being explicitly programmed...
--------------------------------------------------------------------------------

Dokument 2:
Quelle: {'title': 'Maschinelles Lernen – Wikipedia', 'language': 'de', 'source': 'https://de.wikipedia.org/wiki/Maschinelles_Lernen'}
Inhalt (erste 200 Zeichen): 



Maschinelles Lernen – Wikipedia































Zum Inhalt springen







Hauptmenü





Hauptmenü
In die Seitenleiste verschieben
Verbergen



		Navigation
	


HauptseiteThemenpor...
--------------------------------------------------------------------------------

Dokument 3:
Quelle: {'source': 'https://de.wikipedi

In [248]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Ich bin ein großes Sprachmodell, das von Meta entwickelt wurde, und keine Plattform oder ein RAG (Resource Access and Governance). Ich kann Informationen über Maschinelles Lernen und seine Anwendungsmöglichkeiten finden.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [249]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [250]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [251]:
qa_chain = ConversationalRetrievalChain.from_llm(
    ChatOllama(model=model), retriever=retriever, memory=memory, verbose=False
)

In [252]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning ist ein Teilgebiet der maschinellen Lernfähigkeit, bei dem ein Künstler oder ein Computer eine bestimmte Menge von Beispielen verwendet, um zu lernen, welche Variablen einen bestimmten Ausgangswert (Eingabe) und sein gewünschter Ausgangswert (Target) bezeichnen. Dies geschieht durch das Verwerfen einer Vorhersage des Outputs, die auf der Beobachtung dieser Menge von Daten basiert.

Ein klassisches Beispiel für supervised learning ist die künstliche Bilderkennung. Hier werden Muster in Bildern erkannt und verwendet, um Bilder zu verifizieren oder zu analysieren. Ein weiteres Beispiel ist das Gesprächsmodell, bei dem ein Computer eine Frage stellt und auf der Grundlage der Antwort des Nutzers lernen kann, wie er Fragen richtig beantworten soll.

Supervised learning wird häufig in verschiedenen Bereichen eingesetzt, einschließlich:

* Künstliche Bilderkennung (Computer vision)
* Textanalyse
* Sprachmodellierung
* Bewertung von Web-Inhalten

Die Beziehung zwischen Ausga

In [253]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is the rephrased follow-up question:

What types of algorithms are commonly used for supervised learning in various applications?Hier sind einige der häufigsten Algorithmen, die für das supervised- oder künstlichen Lernen eingesetzt werden:

1. **Klassifikationsalgorithmen**:
	* Neuronale Netze (NNs)
	* K-Nearest-Neighbors (k-NN)
	* Support Vector Machines (SVMs)
	* Random Forests
2. **Regression-Algorithmen**:
	* Linear Regression
	* Ridge Regression
	* Lasso Regression
	* ElasticNet
3. **Regressionsalgorithmen**:
	* Decision Trees
	* Gradient Boosting
4. **Clustering-Algorithmen**:
	* Hierarchical Clustering (K-Means)
	* Density-Based Clustering (DBSCAN)
	* k-Medoids

Diese Algorithmen können auf verschiedenen Datenstrukturen und Problembereichen angewendet werden, wie z.B.:

* Klassifikation: Eingabe-Data für eine Kategorisierung zu rechtfertigen.
* Regression: Eingabe-Data um ein bestimmtes Ergebnis vorherzusagen (z.B. Einkaufspreis).
* Clusterierung: Eingabe-Data um Gruppen o